In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
import json

with open("dashboard_data/step3_spend_response.json") as f:
    step3 = json.load(f)

df = pd.read_csv("data/global_ads_performance_dataset.csv")
daily = df.groupby(['date','platform']).agg(ad_spend=('ad_spend','sum'), revenue=('revenue','sum')).reset_index()

platforms = list(step3['elasticity'].keys())
print("Platforms:", platforms)

def revenue_fn(spend, platform):
    b = step3['elasticity'][platform]['beta']
    a = step3['elasticity'][platform]['intercept']
    return np.exp(a) * (spend ** b)

bounds_dict = {}
for p in platforms:
    sub = daily[daily['platform']==p]
    bounds_dict[p] = (sub['ad_spend'].quantile(0.05), sub['ad_spend'].quantile(0.95))
    print(f"{p}: bounds = [{bounds_dict[p][0]:.0f}, {bounds_dict[p][1]:.0f}]")

current_spend = {p: step3['marginal_roas'][p]['avg_daily_spend'] for p in platforms}
total_budget = sum(current_spend.values())
print(f"\nTotal daily budget (giữ nguyên): ${total_budget:.0f}")
print(f"Current allocation: {current_spend}")

def negative_total_revenue(x):
    return -sum(revenue_fn(x[i], platforms[i]) for i in range(len(platforms)))

constraints = {'type': 'eq', 'fun': lambda x: sum(x) - total_budget}
bounds = [bounds_dict[p] for p in platforms]
x0 = [current_spend[p] for p in platforms]

result = minimize(negative_total_revenue, x0, method='SLSQP', bounds=bounds, constraints=constraints)

optimal_spend = dict(zip(platforms, result.x))
optimal_revenue = -result.fun
current_revenue = sum(revenue_fn(current_spend[p], p) for p in platforms)

print("\n=== OPTIMIZATION RESULT ===")
print(f"Current predicted daily revenue: ${current_revenue:,.0f}")
print(f"Optimal predicted daily revenue: ${optimal_revenue:,.0f}")
print(f"Uplift: ${optimal_revenue - current_revenue:,.0f}/day ({(optimal_revenue/current_revenue-1)*100:.1f}%)")
print(f"Annualized uplift estimate: ${(optimal_revenue - current_revenue)*365:,.0f}/year")

print("\n=== ALLOCATION: CURRENT vs OPTIMAL ===")
alloc_comparison = {}
for p in platforms:
    change_pct = (optimal_spend[p]/current_spend[p] - 1) * 100
    print(f"{p}: ${current_spend[p]:,.0f} -> ${optimal_spend[p]:,.0f}  ({change_pct:+.1f}%)")
    alloc_comparison[p] = {
        "current_spend": round(current_spend[p],2),
        "optimal_spend": round(optimal_spend[p],2),
        "change_pct": round(change_pct,1)
    }

output = {
    "total_budget": round(total_budget,2),
    "current_revenue": round(current_revenue,2),
    "optimal_revenue": round(optimal_revenue,2),
    "uplift_daily": round(optimal_revenue - current_revenue,2),
    "uplift_pct": round((optimal_revenue/current_revenue-1)*100,2),
    "uplift_annualized": round((optimal_revenue - current_revenue)*365,2),
    "allocation": alloc_comparison
}
with open("dashboard_data/step4_optimization.json", "w") as f:
    json.dump(output, f, indent=2)
print("\n✅ Saved: dashboard_data/step4_optimization.json")

Platforms: ['Google Ads', 'TikTok Ads', 'Meta Ads']
Google Ads: bounds = [1872, 50111]
TikTok Ads: bounds = [1259, 28814]
Meta Ads: bounds = [506, 17873]

Total daily budget (giữ nguyên): $37804
Current allocation: {'Google Ads': 20681.66, 'TikTok Ads': 10284.57, 'Meta Ads': 6837.86}

=== OPTIMIZATION RESULT ===
Current predicted daily revenue: $162,142
Optimal predicted daily revenue: $204,304
Uplift: $42,163/day (26.0%)
Annualized uplift estimate: $15,389,388/year

=== ALLOCATION: CURRENT vs OPTIMAL ===
Google Ads: $20,682 -> $1,872  (-90.9%)
TikTok Ads: $10,285 -> $28,814  (+180.2%)
Meta Ads: $6,838 -> $7,117  (+4.1%)

✅ Saved: dashboard_data/step4_optimization.json


In [2]:
#constrained_optimization
import pandas as pd
import numpy as np
from scipy.optimize import minimize
import json

with open("dashboard_data/step3_spend_response.json") as f:
    step3 = json.load(f)
with open("dashboard_data/step4_optimization.json") as f:
    step4_theoretical = json.load(f)

df = pd.read_csv("data/global_ads_performance_dataset.csv")
daily = df.groupby(['date','platform']).agg(ad_spend=('ad_spend','sum'), revenue=('revenue','sum')).reset_index()
platforms = list(step3['elasticity'].keys())

def revenue_fn(spend, platform):
    b = step3['elasticity'][platform]['beta']
    a = step3['elasticity'][platform]['intercept']
    return np.exp(a) * (spend ** b)

current_spend = {p: step3['marginal_roas'][p]['avg_daily_spend'] for p in platforms}
total_budget = sum(current_spend.values())
current_revenue = sum(revenue_fn(current_spend[p], p) for p in platforms)

def negative_total_revenue(x):
    return -sum(revenue_fn(x[i], platforms[i]) for i in range(len(platforms)))
    
scenarios = {}
for cap_pct in [15, 25, 35, 50, 100]:  
    cap = cap_pct / 100
    bounds = []
    for p in platforms:
        lo = current_spend[p] * (1 - cap)
        hi = current_spend[p] * (1 + cap)
        bounds.append((max(lo, 0), hi))

    constraints = {'type': 'eq', 'fun': lambda x: sum(x) - total_budget}
    x0 = [current_spend[p] for p in platforms]
    result = minimize(negative_total_revenue, x0, method='SLSQP', bounds=bounds, constraints=constraints)

    opt_spend = dict(zip(platforms, result.x))
    opt_revenue = -result.fun
    uplift_pct = (opt_revenue/current_revenue - 1) * 100

    print(f"\n=== CAP ±{cap_pct}% mỗi kênh ===")
    for p in platforms:
        print(f"  {p}: ${current_spend[p]:,.0f} -> ${opt_spend[p]:,.0f} ({(opt_spend[p]/current_spend[p]-1)*100:+.1f}%)")
    print(f"  Revenue uplift: {uplift_pct:+.1f}% (${(opt_revenue-current_revenue):,.0f}/day)")

    scenarios[f"cap_{cap_pct}pct"] = {
        "cap_pct": cap_pct,
        "allocation": {p: round(opt_spend[p],2) for p in platforms},
        "predicted_revenue": round(opt_revenue,2),
        "uplift_pct": round(uplift_pct,2),
        "uplift_daily": round(opt_revenue-current_revenue,2),
        "uplift_annualized": round((opt_revenue-current_revenue)*365,2)
    }

recommended = scenarios['cap_25pct']
print(f"\n=== KHUYẾN NGHỊ CHÍNH THỨC (Phase 1, cap ±25%) ===")
print(f"Uplift: {recommended['uplift_pct']:+.1f}% doanh thu/ngày, tương đương ${recommended['uplift_annualized']:,.0f}/năm")

output = {
    "current_spend": {p: round(v,2) for p,v in current_spend.items()},
    "current_revenue": round(current_revenue,2),
    "scenarios": scenarios,
    "recommended_scenario": "cap_25pct",
    "theoretical_max_uplift_pct": step4_theoretical['uplift_pct']  
}
with open("dashboard_data/step4b_constrained_scenarios.json", "w") as f:
    json.dump(output, f, indent=2)
print("\n✅ Saved: dashboard_data/step4b_constrained_scenarios.json")


=== CAP ±15% mỗi kênh ===
  Google Ads: $20,682 -> $18,113 (-12.4%)
  TikTok Ads: $10,285 -> $11,827 (+15.0%)
  Meta Ads: $6,838 -> $7,864 (+15.0%)
  Revenue uplift: +3.3% ($5,411/day)

=== CAP ±25% mỗi kênh ===
  Google Ads: $20,682 -> $16,401 (-20.7%)
  TikTok Ads: $10,285 -> $12,856 (+25.0%)
  Meta Ads: $6,838 -> $8,547 (+25.0%)
  Revenue uplift: +5.5% ($8,862/day)

=== CAP ±35% mỗi kênh ===
  Google Ads: $20,682 -> $14,689 (-29.0%)
  TikTok Ads: $10,285 -> $13,884 (+35.0%)
  Meta Ads: $6,838 -> $9,231 (+35.0%)
  Revenue uplift: +7.5% ($12,195/day)

=== CAP ±50% mỗi kênh ===
  Google Ads: $20,682 -> $12,120 (-41.4%)
  TikTok Ads: $10,285 -> $15,427 (+50.0%)
  Meta Ads: $6,838 -> $10,257 (+50.0%)
  Revenue uplift: +10.5% ($16,981/day)

=== CAP ±100% mỗi kênh ===
  Google Ads: $20,682 -> $3,559 (-82.8%)
  TikTok Ads: $10,285 -> $20,569 (+100.0%)
  Meta Ads: $6,838 -> $13,676 (+100.0%)
  Revenue uplift: +19.1% ($30,957/day)

=== KHUYẾN NGHỊ CHÍNH THỨC (Phase 1, cap ±25%) ===
Uplift: +